# 🚀 Cầu Nối SSH Tự Động: Google Colab A100 <-> Antigravity Agent

> **Hướng dẫn:**
> 1. Chọn **Thời gian chạy** -> **Thay đổi loại thời gian chạy** -> Chọn **GPU A100** (hoặc L4) + **RAM cao cấp**.
> 2. Bấm nút **Chạy** (Play) ô mã bên dưới.
> 3. Copy địa chỉ đường hầm `*.trycloudflare.com` dán vào khung chat với Agent.

In [ ]:
# =====================================================================
# KÍCH HOẠT SSH SERVER & CLOUDFLARE TUNNEL (KHÔNG CẦN TÀI KHOẢN)
# =====================================================================
import subprocess, time, re, sys

print("📦 1. Đang cài đặt OpenSSH Server & Cloudflared...")
!apt-get update -qq && apt-get install -y openssh-server -qq > /dev/null
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("🔑 2. Cấu hình SSH Key an toàn cho Agent...")
!mkdir -p /root/.ssh
!echo "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIJbQKc2D2/VQFpFaxO1tqWvvG6/dRc4mQ0lNB8wQsSCV pt@PT" > /root/.ssh/authorized_keys
!chmod 700 /root/.ssh && chmod 600 /root/.ssh/authorized_keys

print("⚙️ 3. Khởi động SSH Service...")
!service ssh restart > /dev/null

print("🌐 4. Đang tạo đường hầm kết nối Cloudflare Tunnel...")
proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "ssh://localhost:22"], stderr=subprocess.PIPE, stdout=subprocess.DEVNULL, text=True)

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 40:
    line = proc.stderr.readline()
    match = re.search(r"https://([a-zA-Z0-9-]+\.trycloudflare\.com)", line)
    if match:
        tunnel_url = match.group(1)
        break
    time.sleep(0.1)

if tunnel_url:
    print("\n" + "="*60)
    print("🎉 KẾT NỐI SẴN SÀNG! HÃY COPY ĐỊA CHỈ NÀY DÁN CHO AGENT:")
    print(f"👉 {tunnel_url}")
    print("="*60 + "\n")
    print("⚠️ Lưu ý: Giữ tab Colab này mở trong suốt buổi làm việc để giữ kết nối.")
else:
    print("❌ Không bắt được URL Cloudflare. Hãy kiểm tra lại kết nối mạng của Colab.")
